In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
torch.manual_seed(42)
np.random.seed(42)
c = 1.0              
sigma = 2.0         
hidden_dim = 64
layers = 4
epochs = 5000
lr = 1e-3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class StandardPINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        return self.net(inputs)

class RFFPINN(nn.Module):
    def __init__(self, sigma):
        super().__init__()
        #  Random Fourier matrix B 
        self.B = nn.Parameter(torch.randn(2, hidden_dim // 2) * sigma, requires_grad=False)
        
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
    def forward(self, x, t):
        inputs = torch.cat([x, t], dim=1)
        # RFF Mapping: cos(2*pi*B*v) and sin(2*pi*B*v)
        proj = 2.0 * np.pi * torch.matmul(inputs, self.B)
        features = torch.cat([torch.cos(proj), torch.sin(proj)], dim=1)
        return self.net(features)



def get_collocation_points(N_f, N_ic, N_bc):
    # PDE collocation points (x, t)
    x_f = torch.empty(N_f, 1).uniform_(-1.0, 1.0).requires_grad_(True).to(device)
    t_f = torch.empty(N_f, 1).uniform_(0.0, 1.0).requires_grad_(True).to(device)
    
    # Initial Condition points (t = 0)
    x_ic = torch.empty(N_ic, 1).uniform_(-1.0, 1.0).to(device)
    t_ic = torch.zeros(N_ic, 1).to(device)
    

    u_ic = torch.exp(-200.0 * x_ic**2)
    

    t_bc = torch.empty(N_bc, 1).uniform_(0.0, 1.0).requires_grad_(True).to(device)
    x_bc_left = -torch.ones(N_bc, 1).requires_grad_(True).to(device)
    x_bc_right = torch.ones(N_bc, 1).requires_grad_(True).to(device)
    
    return x_f, t_f, x_ic, t_ic, u_ic, x_bc_left, x_bc_right, t_bc



def compute_loss(model, x_f, t_f, x_ic, t_ic, u_ic, x_bc_left, x_bc_right, t_bc):
    #  PDE Loss (Physics)
    u_pred = model(x_f, t_f)
    u_t = torch.autograd.grad(u_pred, t_f, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    u_x = torch.autograd.grad(u_pred, x_f, grad_outputs=torch.ones_like(u_pred), create_graph=True)[0]
    f_pred = u_t + c * u_x
    loss_f = torch.mean(f_pred ** 2)
    
    #  IC Loss (Data)
    u_ic_pred = model(x_ic, t_ic)
    loss_ic = torch.mean((u_ic_pred - u_ic) ** 2)
    
    #  BC Loss (Periodic)
    u_left = model(x_bc_left, t_bc)
    u_right = model(x_bc_right, t_bc)
    u_x_left = torch.autograd.grad(u_left, x_bc_left, grad_outputs=torch.ones_like(u_left), create_graph=True)[0]
    u_x_right = torch.autograd.grad(u_right, x_bc_right, grad_outputs=torch.ones_like(u_right), create_graph=True)[0]
    
    loss_bc = torch.mean((u_left - u_right) ** 2) + torch.mean((u_x_left - u_x_right) ** 2)
    
    return loss_f + 10.0 * loss_ic + 10.0 * loss_bc  # Weighting IC and BC heavily




def train_model_adam_lbfgs(model_name, model, adam_epochs=2000, lbfgs_max_iter=1000):
    print(f"\n Training {model_name} ")
    model.train()
    loss_history = []
    

    x_f, t_f, x_ic, t_ic, u_ic, x_bc_left, x_bc_right, t_bc = get_collocation_points(5000, 500, 500)
    
  
    opt_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
    print("Phase 1: Adam Optimization")
    for epoch in range(adam_epochs):
        opt_adam.zero_grad()
        loss = compute_loss(model, x_f, t_f, x_ic, t_ic, u_ic, x_bc_left, x_bc_right, t_bc)
        loss.backward()
        opt_adam.step()
        
        loss_history.append(loss.item())
        if epoch % 500 == 0:
            print(f"  Adam Epoch {epoch}/{adam_epochs} | Loss: {loss.item():.4e}")


    opt_lbfgs = torch.optim.LBFGS(
        model.parameters(), 
        lr=1.0, 
        max_iter=lbfgs_max_iter, 
        max_eval=lbfgs_max_iter * 1.25, 
        tolerance_grad=1e-7, 
        tolerance_change=1e-9, 
        history_size=50, 
        line_search_fn="strong_wolfe" 
    )
    
    print("Phase 2: L-BFGS Optimization")
    lbfgs_iter = 0
    

    def closure():
        nonlocal lbfgs_iter
        opt_lbfgs.zero_grad()
        loss = compute_loss(model, x_f, t_f, x_ic, t_ic, u_ic, x_bc_left, x_bc_right, t_bc)
        loss.backward()
        loss_history.append(loss.item())
        
        if lbfgs_iter % 100 == 0:
            print(f"  L-BFGS Iter {lbfgs_iter} | Loss: {loss.item():.4e}")
        lbfgs_iter += 1
        return loss


    opt_lbfgs.step(closure)
    print(f"  Final L-BFGS Loss: {loss_history[-1]:.4e}")
            
    return loss_history



standard_model = StandardPINN().to(device)
rff_model = RFFPINN(sigma=sigma).to(device)

loss_std = train_model_adam_lbfgs("Standard PINN", standard_model, adam_epochs=2000, lbfgs_max_iter=2000)
loss_rff = train_model_adam_lbfgs("RFF PINN", rff_model, adam_epochs=2000, lbfgs_max_iter=2000)


model_std = standard_model.eval()
model_rff = rff_model.eval()


x_test = torch.linspace(-1, 1, 200).unsqueeze(1).to(device)
t_test_05 = torch.ones_like(x_test) * 0.5
t_test_10 = torch.ones_like(x_test) * 1.0


def exact_gaussian(x, t, c):

    x_shifted = (x - c * t + 1.0) % 2.0 - 1.0
    return np.exp(-200.0 * x_shifted**2)

with torch.no_grad():
    x_np = x_test.cpu().numpy()
    

    u_std_05 = model_std(x_test, t_test_05).cpu().numpy()
    u_rff_05 = model_rff(x_test, t_test_05).cpu().numpy()
    u_exact_05 = exact_gaussian(x_np, 0.5, c)
    

    u_std_10 = model_std(x_test, t_test_10).cpu().numpy()
    u_rff_10 = model_rff(x_test, t_test_10).cpu().numpy()
    u_exact_10 = exact_gaussian(x_np, 1.0, c)


fig, axs = plt.subplots(1, 2, figsize=(14, 5))


axs[0].plot(x_np, u_exact_05, 'k-', linewidth=2, label='Analytical')
axs[0].plot(x_np, u_std_05, 'r--', linewidth=2, label='Standard PINN')
axs[0].plot(x_np, u_rff_05, 'b-.', linewidth=2, label='RFF PINN')
axs[0].set_title("Gaussian Advection at t = 0.5")
axs[0].set_xlabel("x")
axs[0].set_ylabel("u(x,t)")
axs[0].legend()
axs[0].grid(True)


axs[1].plot(x_np, u_exact_10, 'k-', linewidth=2, label='Analytical')
axs[1].plot(x_np, u_std_10, 'r--', linewidth=2, label='Standard PINN')
axs[1].plot(x_np, u_rff_10, 'b-.', linewidth=2, label='RFF PINN')
axs[1].set_title("Gaussian Advection at t = 1.0")
axs[1].set_xlabel("x")
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()


 Training Standard PINN 
Phase 1: Adam Optimization
  Adam Epoch 0/2000 | Loss: 6.9581e-01


/home/neelan/miniconda3/envs/torch/lib/python3.10/site-packages/torch/autograd/graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  Adam Epoch 500/2000 | Loss: 1.1511e-01
  Adam Epoch 1000/2000 | Loss: 5.1414e-02
  Adam Epoch 1500/2000 | Loss: 4.1387e-02
Phase 2: L-BFGS Optimization
  L-BFGS Iter 0 | Loss: 3.1455e-02
  L-BFGS Iter 100 | Loss: 2.3169e-02
  L-BFGS Iter 200 | Loss: 2.0496e-02
  L-BFGS Iter 300 | Loss: 1.8446e-02
  L-BFGS Iter 400 | Loss: 1.6096e-02
  L-BFGS Iter 500 | Loss: 1.4894e-02
  L-BFGS Iter 600 | Loss: 1.3889e-02
  L-BFGS Iter 700 | Loss: 1.2271e-02
  L-BFGS Iter 800 | Loss: 1.0373e-02
  L-BFGS Iter 900 | Loss: 9.0486e-03
  L-BFGS Iter 1000 | Loss: 8.3483e-03
  L-BFGS Iter 1100 | Loss: 6.9842e-03
  L-BFGS Iter 1200 | Loss: 6.2442e-03
  L-BFGS Iter 1300 | Loss: 5.1649e-03
  L-BFGS Iter 1400 | Loss: 4.8438e-03
  L-BFGS Iter 1500 | Loss: 4.3804e-03
  L-BFGS Iter 1600 | Loss: 3.8607e-03
